# 🚢 Titanic Survival Prediction
## Machine Learning Lab — Classification Problem

**Dataset:** Titanic Passenger Dataset (Kaggle / public GitHub mirror)  
**Source:** https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv  
**Task:** Binary Classification — Predict whether a passenger survived the Titanic disaster

---

### Problem Description

The Titanic dataset contains information about 891 passengers aboard the RMS Titanic, which sank in April 1912. Each row represents one passenger with features such as age, sex, ticket class, fare, and number of family members aboard.

**This is a Binary Classification problem.**

- **Target variable:** `Survived` (1 = survived, 0 = did not survive)
- **Goal:** Build a model that learns patterns from passenger attributes to predict survival
- **Why it matters:** Understanding which factors most influenced survival can reveal historical insights about evacuation priorities (e.g., "women and children first") and socioeconomic disparities

The model is expected to learn decision boundaries based on features like passenger class, sex, age, and embarkation point to classify unseen passengers as survivors or non-survivors.


In [1]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np

print("Libraries loaded successfully ✓")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries loaded successfully ✓
Pandas version: 2.2.3
NumPy version: 2.1.3


In [2]:
# Step 2: Load the dataset
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Source: {url}")


Dataset loaded successfully!
Source: https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv


In [3]:
# Step 3: Display dataset shape
rows, cols = df.shape
print(f" Dataset Shape: {rows} rows × {cols} columns")
print(f"   → {rows} passengers (observations)")
print(f"   → {cols} features (variables)")


 Dataset Shape: 891 rows × 12 columns
   → 891 passengers (observations)
   → 12 features (variables)


In [4]:
# Step 4: Preview the first 5 rows
print("First 5 rows of the dataset:")
df.head()


First 5 rows of the dataset:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Step 5: Check column names and data types
print("Column Names and Data Types:")
print("=" * 45)
print(f"{'Column':<15} {'Dtype':<12} {'Non-Null Count'}")
print("-" * 45)
for col in df.columns:
    non_null = df[col].notna().sum()
    print(f"{col:<15} {str(df[col].dtype):<12} {non_null}/{len(df)}")


Column Names and Data Types:
Column          Dtype        Non-Null Count
---------------------------------------------
PassengerId     int64        891/891
Survived        int64        891/891
Pclass          int64        891/891
Name            object       891/891
Sex             object       891/891
Age             float64      714/891
SibSp           int64        891/891
Parch           int64        891/891
Ticket          object       891/891
Fare            float64      891/891
Cabin           object       204/891
Embarked        object       889/891


In [6]:
# Step 6: Full dataset info summary
print("Full Dataset Info:")
df.info()


Full Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [7]:
# Step 7: Statistical summary of numeric columns
print("Statistical Summary:")
df.describe()


Statistical Summary:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [8]:
# Step 8: Check for missing values
print("Missing Values per Column:")
print("=" * 35)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(missing_df.to_string())
print(f"\n→ Columns with missing data: {len(missing_df)}")


Missing Values per Column:
          Missing Count  Missing %
Age                 177      19.87
Cabin               687      77.10
Embarked              2       0.22

→ Columns with missing data: 3


In [9]:
# Step 9: Target variable distribution
print("Target Variable: 'Survived'")
print("=" * 35)
counts = df['Survived'].value_counts()
pcts = df['Survived'].value_counts(normalize=True) * 100
print(f"  Did NOT Survive (0): {counts[0]} passengers ({pcts[0]:.1f}%)")
print(f"  Survived        (1): {counts[1]} passengers ({pcts[1]:.1f}%)")
print(f"\n→ This is a moderately imbalanced binary classification task.")


Target Variable: 'Survived'
  Did NOT Survive (0): 549 passengers (61.6%)
  Survived        (1): 342 passengers (38.4%)

→ This is a moderately imbalanced binary classification task.


---

## 📝 Summary

| Item | Detail |
|------|--------|
| **Dataset** | Titanic Passenger Data |
| **Source** | Kaggle / GitHub (datasciencedojo) |
| **Rows** | 891 passengers |
| **Columns** | 12 features |
| **Problem Type** | Binary Classification |
| **Target Variable** | `Survived` (0 or 1) |
| **Missing Data** | Age (19.9%), Cabin (77.1%), Embarked (0.2%) |

### Next Steps (Future Labs)
1. **Data Preprocessing** — Handle missing values, encode categorical variables, drop irrelevant columns (Name, Ticket, Cabin)
2. **Feature Engineering** — Extract title from Name, create family size feature
3. **Train/Test Split** — 80% train / 20% test
4. **Model Training** — Try Logistic Regression, Decision Tree, Random Forest
5. **Evaluation** — Accuracy, Precision, Recall, F1-Score, Confusion Matrix
